# ExPECA: three SDR containers (one gNB and two UEs)

This notebook extends the ExPECA [`2sdr_2containers.ipynb`](https://github.com/KTH-EXPECA/examples/blob/main/sdr/2sdr_2containers.ipynb) pattern to reserve three SDRs and three workers, then create one RF-attached container for each role:

- one **gNB** container,
- one **UE 1** container, and
- one **UE 2** container.

> This notebook provisions the hosts and SDR-facing networks. Like the source notebook, it uses the general `samiemostafavi/sshd-dind-oai` image; the gNB/UE role is selected when OAI is configured and started inside each container. It does not create the 5G Core or start the OAI processes.

Derived from KTH-EXPECA/examples commit `e90e57f2f8d3f6e226f60b9f976de976dee44e2a`.

## 1. Set input parameters

Edit this cell before running anything. The third worker and SDR are the additional inputs compared with the two-SDR example. Each worker needs one free 10 Gb/s interface for its SDR and one additional free interface for the public network.

In [ ]:
# ExPECA project and reservation duration
project = "YOUR-EXPECA-PROJECT"
lease_days = 7

# gNB host
gnb_sdr_name = "sdr-01"
gnb_worker_name = "worker-02"

# UE 1 host
ue1_sdr_name = "sdr-02"
ue1_worker_name = "worker-03"

# UE 2 host (the extra SDR/worker pair)
ue2_sdr_name = "sdr-03"
ue2_worker_name = "worker-04"

# Container-side addresses on the three separate SDR SFP networks.
# These are host addresses, not the SDR device addresses used by UHD.
gnb_sdr_host_ip = "10.30.10.121/24"
ue1_sdr_host_ip = "10.30.10.122/24"
ue2_sdr_host_ip = "10.30.10.123/24"

container_image = "samiemostafavi/sshd-dind-oai"
container_password = "expeca"
public_gateway = "130.237.11.97"
dns_ip = "8.8.8.8"

# Colab default. Change this when running Jupyter elsewhere.
openrc_path = f"/content/{project}-openrc.sh"

radio_hosts = [
    {
        "role": "gnb",
        "container_name": "oai-gnb",
        "sdr_name": gnb_sdr_name,
        "worker_name": gnb_worker_name,
        "sdr_host_ip": gnb_sdr_host_ip,
    },
    {
        "role": "ue1",
        "container_name": "oai-ue1",
        "sdr_name": ue1_sdr_name,
        "worker_name": ue1_worker_name,
        "sdr_host_ip": ue1_sdr_host_ip,
    },
    {
        "role": "ue2",
        "container_name": "oai-ue2",
        "sdr_name": ue2_sdr_name,
        "worker_name": ue2_worker_name,
        "sdr_host_ip": ue2_sdr_host_ip,
    },
]

## 2. Authentication and dependencies

Download the project OpenRC file from the [ExPECA API access page](https://testbed.expeca.proj.kth.se/project/api_access/openrc/), place it at `openrc_path`, and run the next cell. The password is requested interactively and is not stored in the notebook.

In [ ]:
import os
import re
from getpass import getpass

with open(openrc_path, "r") as openrc_file:
    script_content = openrc_file.read()

pattern = r'export\s+(\w+)\s*=\s*("[^"]+"|[^"\n]+)'
for name, value in re.findall(pattern, script_content):
    os.environ[name] = value.strip('"')

os.environ["OS_PASSWORD"] = getpass("Enter your ExPECA password: " )

In [ ]:
!pip uninstall -q -y moviepy
!pip install -q jedi
!pip install -q git+https://github.com/KTH-EXPECA/python-chi

In [ ]:
import ipaddress
import json

from loguru import logger
import chi.container
import chi.network
from chi.expeca import (
    get_available_publicips,
    get_container_status,
    get_radio_interfaces,
    get_segment_ids,
    get_worker_interfaces,
    list_reservations,
    reserve,
    show_reservation_byname,
)

## 3. Validate the requested topology

Fail early if a name or address was accidentally reused. This notebook deliberately requires three distinct workers because that is the requested topology.

In [ ]:
def require_unique(field):
    values = [host[field] for host in radio_hosts]
    if len(values) != len(set(values)):
        raise ValueError(f"{field} values must be unique: {values}")

if project == "YOUR-EXPECA-PROJECT":
    raise ValueError("Set project in the first code cell before continuing.")

if [host["role"] for host in radio_hosts] != ["gnb", "ue1", "ue2"]:
    raise ValueError("radio_hosts must contain exactly one gNB followed by UE 1 and UE 2.")

for field in ("container_name", "sdr_name", "worker_name", "sdr_host_ip"):
    require_unique(field)

for host in radio_hosts:
    interface = ipaddress.ip_interface(host["sdr_host_ip"])
    if interface.ip == interface.network.network_address or interface.ip == interface.network.broadcast_address:
        raise ValueError(f"Invalid host address for {host['role']}: {interface}")

logger.success("Validated one gNB plus two UEs on three distinct SDRs and workers.")

## 4. Check all three SDRs

Both reported ports should be up. Stop and contact ExPECA support if a required port remains down.

In [ ]:
for host in radio_hosts:
    sdr_status = get_radio_interfaces(host["sdr_name"])
    logger.info(f"{host['role']} / {host['sdr_name']}:\n{json.dumps(sdr_status, indent=4)}")
    for port, status in sdr_status.items():
        if status["linkstate"] == "Up":
            logger.success(f"Port {port} on {host['sdr_name']} is up.")
        else:
            logger.warning(f"Port {port} on {host['sdr_name']} is {status['linkstate']}.")

## 5. Reserve three SDRs and three workers

For each SDR, reserve both physical port segments using the naming convention from the upstream notebook. Then reserve its worker and record the device reservation ID used by Zun. Re-running this cell reuses matching reservations visible to the project.

In [ ]:
def reserve_sdr_ports(sdr_name):
    segment_ids = get_segment_ids(sdr_name)
    leases = {}
    for port_name in ("rj45", "sfp"):
        lease_name = f"{sdr_name}-{port_name}-lease"
        lease = show_reservation_byname(lease_name)
        if not lease:
            lease = reserve({
                "type": "network",
                "name": f"{sdr_name}-{port_name}",
                "net_name": f"{sdr_name}-{port_name}",
                "segment_id": segment_ids[port_name],
                "duration": {"days": lease_days, "hours": 0},
            })
        leases[port_name] = lease
    return leases

def reserve_worker(worker_name):
    lease = show_reservation_byname(f"{worker_name}-lease")
    if not lease:
        lease = reserve({
            "type": "device",
            "name": worker_name,
            "duration": {"days": lease_days, "hours": 0},
        })
    return lease, lease["reservations"][0]["id"]

for host in radio_hosts:
    host["sdr_leases"] = reserve_sdr_ports(host["sdr_name"])
    host["worker_lease"], host["worker_reservation_id"] = reserve_worker(host["worker_name"])

print(json.dumps(list_reservations(brief=True), indent=4))

## 6. Create the gNB and two UE containers

The helper mirrors the upstream implementation: it selects a free 10 Gb/s worker interface for the SDR SFP network and a second free interface for the public network. It refuses to overwrite an existing container with the same name.

In [ ]:
def choose_worker_interfaces(worker_name):
    worker_data = get_worker_interfaces(worker_name)
    if not worker_data:
        raise RuntimeError(f"No interface data returned for {worker_name}.")

    interfaces = list(worker_data.values())[0]
    free_10g = []
    free_other = []
    for interface_name, details in interfaces.items():
        if details["connections"]:
            continue
        if str(details["speed"]) == "10000":
            free_10g.append(interface_name)
        else:
            free_other.append(interface_name)

    logger.info(f"Free 10G interfaces on {worker_name}: {free_10g}")
    logger.info(f"Free other interfaces on {worker_name}: {free_other}")

    if not free_10g:
        logger.info(json.dumps(interfaces, indent=4))
        raise RuntimeError(f"No free 10G interface found on {worker_name}.")

    sdr_interface = free_10g.pop(0)
    if free_other:
        public_interface = free_other.pop(0)
    elif free_10g:
        public_interface = free_10g.pop(0)
    else:
        logger.info(json.dumps(interfaces, indent=4))
        raise RuntimeError(f"No second free interface found on {worker_name}.")

    return sdr_interface, public_interface

def create_radio_container(host):
    if get_container_status(host["container_name"]):
        raise RuntimeError(
            f"Container {host['container_name']} already exists; inspect or remove it before retrying."
        )

    available_public_ips = get_available_publicips()
    if not available_public_ips:
        raise RuntimeError("There are no available public IPs to reserve.")
    public_ip = available_public_ips[0]

    sdr_interface, public_interface = choose_worker_interfaces(host["worker_name"])
    sdr_network = chi.network.get_network(f"{host['sdr_name']}-sfp-net")
    public_network = chi.network.get_network("serverpublic")

    chi.container.create_container(
        name=host["container_name"],
        image=container_image,
        reservation_id=host["worker_reservation_id"],
        environment={
            "DNS_IP": dns_ip,
            "GATEWAY_IP": public_gateway,
            "PASS": container_password,
            "OAI_ROLE": host["role"],
        },
        mounts=[],
        nets=[
            {"network": public_network["id"]},
            {"network": sdr_network["id"]},
        ],
        labels={
            "networks.1.interface": public_interface,
            "networks.1.ip": f"{public_ip}/27",
            "networks.2.interface": sdr_interface,
            "networks.2.ip": host["sdr_host_ip"],
            "capabilities.privileged": "true",
        },
    )
    chi.container.wait_for_active(host["container_name"])
    logger.success(
        f"Created {host['container_name']} ({host['role']}) on {host['worker_name']}; "
        f"public address {public_ip}, SDR {host['sdr_name']}."
    )

    install_command = "apt-get update && apt-get install -y iproute2 iputils-ping iperf3"
    result = chi.container.execute(
        container_ref=host["container_name"],
        command=(
            "curl -s -X POST -H \"Content-Type: application/json\" "
            f"-d '{{\"cmd\": \"{install_command}\"}}' http://localhost:50505/"
        ),
    )
    logger.info(result)

    return {
        "role": host["role"],
        "container": host["container_name"],
        "worker": host["worker_name"],
        "sdr": host["sdr_name"],
        "public_ip": public_ip,
        "sdr_host_ip": host["sdr_host_ip"],
    }

In [ ]:
deployments = []
for host in radio_hosts:
    deployments.append(create_radio_container(host))

print(json.dumps(deployments, indent=4))

## 7. Verify container and SDR access

The checks below confirm that each container is active and print its interfaces. They do not prove RF operation. Afterward, log in to each container and run `uhd_find_devices` to discover the address of its attached SDR.

In [ ]:
for deployment in deployments:
    status = get_container_status(deployment["container"])
    if not status:
        raise RuntimeError(f"{deployment['container']} is not active.")

    logger.success(f"{deployment['container']} is active: {status}")
    result = chi.container.execute(
        container_ref=deployment["container"],
        command=(
            "curl -s -X POST -H \"Content-Type: application/json\" "
            "-d '{\"cmd\": \"ip -br address\"}' http://localhost:50505/"
        ),
    )
    logger.info(f"{deployment['role']} interfaces:\n{result}")

## 8. OAI role handoff

Provisioning is complete when all three containers are active and each can discover only its intended SDR. To turn this into a working 5G system:

1. Provide a 5G Core and N2/N3 reachability to `oai-gnb` (or run the core inside that container, if that is the chosen ExPECA design).
2. Register **two distinct subscribers** in the core. UE 1 and UE 2 must not share an IMSI; keep each UE's IMSI/key/OPC consistent with the core database.
3. In each container, run `uhd_find_devices` and use the returned SDR address in the corresponding OAI configuration. The `10.30.10.121-123` values above are container-side interface addresses, not USRP addresses.
4. Start the core, then the gNB in `oai-gnb`, then the nrUE process in `oai-ue1` and `oai-ue2`. Both UEs must use the same band, center frequency, numerology, and bandwidth as the gNB.
5. Confirm that each UE receives its own PDU-session interface/address and can reach the intended edge endpoint before starting RP-Bench.

The role-specific ExPECA images shown in the older [`sdr5g_latency_measurement_setup.ipynb`](https://github.com/KTH-EXPECA/examples/blob/main/wireless-pr3d/sdr5g_latency_measurement_setup.ipynb) can be adopted later, once the 5GC and exact radio configuration are fixed.